In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# %cd /content/drive/MyDrive/NER

In [ ]:
# !pip install transformers seqeval[gpu]

## **Fine-tuning BERT for named-entity recognition**

In this notebook, we are going to use **BertForTokenClassification** which is included in the [Transformers library](https://github.com/huggingface/transformers) by HuggingFace. This model has BERT as its base architecture, with a token classification head on top, allowing it to make predictions at the token level, rather than the sequence level. Named entity recognition is typically treated as a token classification problem, so that's what we are going to use it for.

This tutorial uses the idea of **transfer learning**, i.e. first pretraining a large neural network in an unsupervised way, and then fine-tuning that neural network on a task of interest. In this case, BERT is a neural network pretrained on 2 tasks: masked language modeling and next sentence prediction. Now, we are going to fine-tune this network on a NER dataset. Fine-tuning is supervised learning, so this means we will need a labeled dataset.

If you want to know more about BERT, I suggest the following resources:
* the original [paper](https://arxiv.org/abs/1810.04805)
* Jay Allamar's [blog post](http://jalammar.github.io/illustrated-bert/) as well as his [tutorial](http://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/)
* Chris Mccormick's [Youtube channel](https://www.youtube.com/channel/UCoRX98PLOsaN8PtekB9kWrw)
* Abbishek Kumar Mishra's [Youtube channel](https://www.youtube.com/user/abhisheksvnit)

The following notebook largely follows the same structure as the tutorials by Abhishek Kumar Mishra. For his tutorials on the Transformers library, see his [Github repository](https://github.com/abhimishra91/transformers-tutorials).

NOTE: this notebook assumes basic knowledge about deep learning, BERT, and native PyTorch. If you want to learn more Python, deep learning and PyTorch, I highly recommend cs231n by Stanford University and the FastAI course by Jeremy Howard et al. Both are freely available on the web.  

Now, let's move on to the real stuff!

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertConfig, BertForTokenClassification

As deep learning can be accellerated a lot using a GPU instead of a CPU, make sure you can run this notebook in a GPU runtime (which Google Colab provides for free! - check "Runtime" - "Change runtime type" - and set the hardware accelerator to "GPU").

We can set the default device to GPU using the following code (if it prints "cuda", it means the GPU has been recognized):

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

#### **Downloading and preprocessing the data**
Named entity recognition (NER) uses a specific annotation scheme, which is defined (at least for European languages) at the *word* level. An annotation scheme that is widely used is called **[IOB-tagging](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging)**, which stands for Inside-Outside-Beginning. Each tag indicates whether the corresponding word is *inside*, *outside* or at the *beginning* of a specific named entity. The reason this is used is because named entities usually comprise more than 1 word.

Let's have a look at an example. If you have a sentence like "Barack Obama was born in Hawaï", then the corresponding tags would be   [B-PERS, I-PERS, O, O, O, B-GEO]. B-PERS means that the word "Barack" is the beginning of a person, I-PERS means that the word "Obama" is inside a person, "O" means that the word "was" is outside a named entity, and so on. So one typically has as many tags as there are words in a sentence.

So if you want to train a deep learning model for NER, it requires that you have your data in this IOB format (or similar formats such as [BILOU](https://stackoverflow.com/questions/17116446/what-do-the-bilou-tags-mean-in-named-entity-recognition)). There exist many annotation tools which let you create these kind of annotations automatically (such as Spacy's [Prodigy](https://prodi.gy/), [Tagtog](https://docs.tagtog.net/) or [Doccano](https://github.com/doccano/doccano)). You can also use Spacy's [biluo_tags_from_offsets](https://spacy.io/api/goldparse#biluo_tags_from_offsets) function to convert annotations at the character level to IOB format.

Here, we will use a NER dataset from [Kaggle](https://www.kaggle.com/namanj27/ner-dataset) that is already in IOB format. One has to go to this web page, download the dataset, unzip it, and upload the csv file to this notebook. Let's print out the first few rows of this csv file:

In [ ]:
DRIVE = False # change if running on Google Colab

if DRIVE:
    train_data = pd.read_csv('/content/drive/MyDrive/NER/train_ner_doctored.csv')
    eval_data = pd.read_csv('/content/drive/MyDrive/NER/eval_ner_doctored.csv')
    test_data = pd.read_csv('/content/drive/MyDrive/NER/test_ner_doctored.csv')
else:
    # train_data = pd.read_csv('dataset_formation/data/train_ner_doctored.csv')
    # eval_data = pd.read_csv('dataset_formation/data/eval_ner_doctored.csv')
    # test_data = pd.read_csv('dataset_formation/data/test_ner_doctored.csv')
    
    # new data
    train_data = pd.read_csv('dataset_formation/final/train_original_data_v1.0.csv')
    eval_data = pd.read_csv('dataset_formation/final/eval.csv')
    test_data = pd.read_csv('dataset_formation/final/test.csv')

In [ ]:
train_data.head(200)

In [ ]:
test_data.term.unique()

In [ ]:
len(test_data.term.unique())

array(['khachkar', 'tax', 'coffin', 'coffin', 'donation',
'construction protocol', 'Construction protocol',
'reservation', nan], dtype=object)

In [ ]:
train_data.term.unique()

In [ ]:
eval_data.term.unique()

We create 2 dictionaries: one that maps individual tags to indices, and one that maps indices to their individual tags. This is necessary in order to create the labels (as computers work with numbers = indices, rather than words = tags) - see further in this notebook.

In [ ]:
labels_to_ids = {k: v for v, k in enumerate(['O','B-INSC','I-INSC'])}
ids_to_labels = {v: k for v, k in enumerate(['O','B-INSC','I-INSC'])}
labels_to_ids

Let's verify that a random sentence and its corresponding tags are correct:

In [ ]:
train_data.iloc[11].sentence

In [ ]:
train_data.iloc[11].word_labels

### Plots

In [ ]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

train_data["split"] = "Train"
eval_data["split"] = "Eval"
test_data["split"] = "Test"
data_all = pd.concat([train_data, eval_data, test_data], ignore_index=True)

# Determine if each row has a non-null term
data_all["has_term"] = data_all["term"].notna()

summary = (
    data_all.groupby(["split", "has_term"])
    .size()
    .reset_index(name="count")
)

summary_pivot = summary.pivot(index="split", columns="has_term", values="count").fillna(0)
summary_pivot.columns = ["No Term", "Has Term"]
summary_pivot["Total"] = summary_pivot.sum(axis=1)
summary_pivot = summary_pivot.astype(int).reset_index()

sns.set_theme(style="whitegrid", palette="colorblind")
plt.figure(figsize=(7, 5))
sns.barplot(data=summary, x="split", y="count", hue="has_term")


plt.title("Samples with and without a Term per Dataset Split", fontsize=14, weight="bold")
plt.xlabel("Dataset Split", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.legend(title="Has Term", labels=["No", "Yes"])
plt.tight_layout()

os.makedirs("images", exist_ok=True)
plot_path = "images/samples_with_and_without_term_v1.0.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")

print(f"Figure saved to: {plot_path}")

plt.show()

In [ ]:
# generate the LaTex
latex_table = summary_pivot.to_latex(
    index=False,
    caption="Number of samples with and without a term per dataset split.",
    label="tab:term-presence",
    column_format="lccc",
    bold_rows=True,
    escape=False
)

print(latex_table)

In [ ]:
# \begin{table}
# \caption{Number of samples with and without a term per dataset split.}
# \label{tab:term-presence}
# \begin{tabular}{lccc}
# \toprule
# split & No Term & Has Term & Total \\
# \midrule
# Eval & 252 & 17 & 269 \\
# Test & 254 & 17 & 271 \\
# Train & 1131 & 76 & 1207 \\
# \bottomrule
# \end{tabular}
# \end{table}

#### **Preparing the dataset and dataloader**

Now that our data is preprocessed, we can turn it into PyTorch tensors such that we can provide it to the model. Let's start by defining some key variables that will be used later on in the training/evaluation process:

In [ ]:
import random
import numpy as np
import torch

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Make CUDA deterministic (slightly slower)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
MAX_LEN = 128 # might be a bit small
TRAIN_BATCH_SIZE = 32 # 4
VALID_BATCH_SIZE = 32 # 2
EPOCHS = 3 # 1
LEARNING_RATE = 3e-5 # 1e-05 -> For small datasets, you might even try 2e-5 or 3e-5.
MAX_GRAD_NORM = 2

BASE_MODEL = "bert-base-multilingual-uncased" # 'xlm-roberta-base' # "xlm-roberta-base" #"daviddallakyan2005/armenian-ner" # 'bert-base-multilingual-uncased'
# hye
from transformers import AutoTokenizer
# tokenizer = BertTokenizerFast.from_pretrained(BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

A tricky part of NER with BERT is that BERT relies on **wordpiece tokenization**, rather than word tokenization. This means that we should also define the labels at the wordpiece-level, rather than the word-level!

For example, if you have word like "Washington" which is labeled as "b-gpe", but it gets tokenized to "Wash", "##ing", "##ton", then one approach could be to handle this by only train the model on the tag labels for the first word piece token of a word (i.e. only label "Wash" with "b-gpe"). This is what was done in the original BERT paper, see Github discussion [here](https://github.com/huggingface/transformers/issues/64#issuecomment-443703063).

Note that this is a **design decision**. You could also decide to propagate the original label of the word to all of its word pieces and let the model train on this. In that case, the model should be able to produce the correct labels for each individual wordpiece. This was done in [this NER tutorial with BERT](https://github.com/chambliss/Multilingual_NER/blob/master/python/utils/main_utils.py#L118). Another design decision could be to give the first wordpiece of each word the original word label, and then use the label “X” for all subsequent subwords of that word. All of them seem to lead to good performance.

Below, we define a regular PyTorch [dataset class](https://pytorch.org/docs/stable/data.html) (which transforms examples of a dataframe to PyTorch tensors). Here, each sentence gets tokenized, the special tokens that BERT expects are added, the tokens are padded or truncated based on the max length of the model, the attention mask is created and the labels are created based on the dictionary which we defined above. Word pieces that should be ignored have a label of -100 (which is the default `ignore_index` of PyTorch's [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)).

For more information about BERT's inputs, see [here](https://huggingface.co/transformers/glossary.html).








In [ ]:
class dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = self.data.sentence[index].strip().split()
        word_labels = self.data.word_labels[index].split(",")

        encoding = self.tokenizer(
            sentence,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        # Create aligned labels
        labels = [labels_to_ids[label] for label in word_labels]
        word_ids = encoding.word_ids(batch_index=0)

        # Initialize with -100 - but for hye and other models roberta -> this is some other code
        encoded_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                encoded_labels.append(-100)
            elif word_idx != previous_word_idx:
                encoded_labels.append(labels[word_idx])
            else:
                encoded_labels.append(-100)
            previous_word_idx = word_idx

        # Convert everything to tensors
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(encoded_labels, dtype=torch.long)

        return item

    def __len__(self):
        return len(self.data)

Now, based on the class we defined above, we can create 2 datasets, one for training and one for testing. Let's use a 80/20 split:

In [ ]:
training_set = dataset(train_data, tokenizer, MAX_LEN)


In [ ]:
training_set[0]

In [ ]:

testing_set = dataset(test_data, tokenizer, MAX_LEN)
validation_set = dataset(eval_data, tokenizer, MAX_LEN)

In [ ]:
train_data.iloc[0]

Let's have a look at the first training example:

In [ ]:
testing_set[0]

Let's verify that the input ids and corresponding targets are correct:

In [ ]:
train_data.head()

In [ ]:
for token, label in zip(tokenizer.convert_ids_to_tokens(training_set[9]["input_ids"]), training_set[9]["labels"]):
  print('{0:10}  {1}'.format(token, label))

-100 = ignore index -> loss is not computed for these positions (CrossEntropyLoss skips them).

Now, let's define the corresponding PyTorch dataloaders:

In [ ]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

valid_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)
validation_loader = DataLoader(validation_set, **test_params)

#### **Defining the model**

Here we define the model, BertForTokenClassification, and load it with the pretrained weights of "bert-base-uncased". The only thing we need to additionally specify is the number of labels (as this will determine the architecture of the classification head).

Note that only the base layers are initialized with the pretrained weights. The token classification head of top has just randomly initialized weights, which we will train, together with the pretrained weights, using our labelled dataset. This is also printed as a warning when you run the code cell below.

Then, we move the model to the GPU.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, num_labels=len(labels_to_ids))
model.to(device)

#### **Training the model**

Before training the model, let's perform a sanity check, which I learned thanks to Andrej Karpathy's wonderful [cs231n course](http://cs231n.stanford.edu/) at Stanford (see also his [blog post about debugging neural networks](http://karpathy.github.io/2019/04/25/recipe/)). The initial loss of your model should be close to -ln(1/number of classes) = -ln(1/17) = 2.83.

Why? Because we are using cross entropy loss. The cross entropy loss is defined as -ln(probability score of the model for the correct class). In the beginning, the weights are random, so the probability distribution for all of the classes for a given token will be uniform, meaning that the probability for the correct class will be near 1/17. The loss for a given token will thus be -ln(1/17). As PyTorch's [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) (which is used by `BertForTokenClassification`) uses *mean reduction* by default, it will compute the mean loss for each of the tokens in the sequence for which a label is provided.

Let's verify this:



In [ ]:
# inputs = training_set[2]
# input_ids = inputs["input_ids"].unsqueeze(0)
# attention_mask = inputs["attention_mask"].unsqueeze(0)
# labels = inputs["labels"].unsqueeze(0)

# input_ids = input_ids.to(device)
# attention_mask = attention_mask.to(device)
# labels = labels.to(device)

# outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
# initial_loss = outputs[0]
# initial_loss

This looks good. Let's also verify that the logits of the neural network have a shape of (batch_size, sequence_length, num_labels):

In [ ]:
# tr_logits = outputs[1]
# tr_logits.shape

In [ ]:
import torch
from tqdm import tqdm

def predict(model, dataloader, device, ids_to_labels):
    """
    Generate predictions and ground truth labels for a token classification model.

    Args:
        model: Trained HuggingFace model (e.g., BertForTokenClassification)
        dataloader: DataLoader providing batches with 'input_ids', 'attention_mask', and 'labels'
        device: torch.device('cuda' or 'cpu')
        ids_to_labels: dict mapping label IDs to label strings

    Returns:
        labels_list: List[List[str]] — true labels per sentence (excluding padding)
        predictions_list: List[List[str]] — predicted labels per sentence (excluding padding)
    """
    model.eval()
    labels_list = []
    predictions_list = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            for i in range(labels.size(0)):
                sentence_labels = labels[i]               # (seq_len,)
                sentence_logits = logits[i]               # (seq_len, num_labels)
                sentence_preds = torch.argmax(sentence_logits, dim=-1)

                # Filter out ignored tokens (-100)
                active_mask = sentence_labels != -100
                filtered_labels = sentence_labels[active_mask]
                filtered_preds = sentence_preds[active_mask]

                labels_list.append([
                    ids_to_labels[id.item()] for id in filtered_labels
                ])
                predictions_list.append([
                    ids_to_labels[id.item()] for id in filtered_preds
                ])

    return labels_list, predictions_list

Next, we define the optimizer. Here, we are just going to use Adam with a default learning rate. One can also decide to use more advanced ones such as AdamW (Adam with weight decay fix), which is [included](https://huggingface.co/transformers/main_classes/optimizer_schedules.html) in the Transformers repository, and a learning rate scheduler, but we are not going to do that here.

In [ ]:
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)
# optimizer = torch.optim.AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

Now let's define a regular PyTorch training function. It is partly based on [a really good repository about multilingual NER](https://github.com/chambliss/Multilingual_NER/blob/master/python/utils/main_utils.py#L344).

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import torch
from seqeval.metrics import classification_report, f1_score
import copy

def train_one_epoch(epoch, model, training_loader, validation_loader, optimizer, device, ids_to_labels, eval_every=100):
    tr_loss, tr_accuracy = 0, 0
    nb_tr_steps = 0

    train_step_losses = []   # training loss every 100 steps
    eval_step_losses = []    # validation loss every 100 steps

    best_f1 = 0.0
    best_model_state = None
    best_report = None

    model.train()
    progress_bar = tqdm(training_loader, desc=f"Epoch {epoch} [Training]", leave=False)

    for idx, batch in enumerate(progress_bar):
        ids = batch['input_ids'].to(device, dtype=torch.long)
        mask = batch['attention_mask'].to(device, dtype=torch.long)
        labels = batch['labels'].to(device, dtype=torch.long)

        outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        tr_loss += loss.item()
        nb_tr_steps += 1

        # Accuracy
        flattened_targets = labels.view(-1)
        active_logits = logits.view(-1, model.num_labels)
        flattened_predictions = torch.argmax(active_logits, axis=1)
        active_mask = flattened_targets != -100

        labels_filtered = torch.masked_select(flattened_targets, active_mask)
        preds_filtered = torch.masked_select(flattened_predictions, active_mask)

        tmp_tr_accuracy = accuracy_score(
            labels_filtered.cpu().numpy(),
            preds_filtered.cpu().numpy()
        )
        tr_accuracy += tmp_tr_accuracy

        # Gradient clipping + backward
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Record train loss every eval_every steps
        if idx % eval_every == 0 and idx > 0:
            avg_loss = tr_loss / nb_tr_steps
            train_step_losses.append(avg_loss)

            # Run quick evaluation every eval_every steps
            model.eval()
            eval_loss, eval_steps = 0, 0
            with torch.no_grad():
                for v_idx, v_batch in enumerate(validation_loader):
                    # if v_idx >= 5:  # limit evaluation to 5 batches to save time
                    #     break
                    v_ids = v_batch['input_ids'].to(device, dtype=torch.long)
                    v_mask = v_batch['attention_mask'].to(device, dtype=torch.long)
                    v_labels = v_batch['labels'].to(device, dtype=torch.long)

                    v_outputs = model(input_ids=v_ids, attention_mask=v_mask, labels=v_labels)
                    eval_loss += v_outputs.loss.item()
                    eval_steps += 1

                # Predict on validation data
                labels, predictions = predict(model, validation_loader, device, ids_to_labels)
                report = classification_report(labels, predictions, digits=4, output_dict=True)
                current_f1 = report["macro avg"]["f1-score"]

                print(f"\n[Step {idx}] F1 (macro): {current_f1:.4f}")
                eval_step_losses.append(eval_loss / eval_steps)

                # Save best model
                if current_f1 > best_f1:
                    best_f1 = current_f1
                    best_model_state = copy.deepcopy(model.state_dict())
                    best_report = report
                    print(f"  ✅ New best model (F1={best_f1:.4f}) saved.")

            model.train()

        # Update tqdm
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{tmp_tr_accuracy:.4f}'
        })

    epoch_loss = tr_loss / nb_tr_steps
    epoch_acc = tr_accuracy / nb_tr_steps

    print(f"\nEpoch {epoch} Summary:")
    print(f"  Training loss: {epoch_loss:.4f}")
    print(f"  Training accuracy: {epoch_acc:.4f}")
    if best_f1 > 0:
        print(f"  Best validation F1: {best_f1:.4f}")

    return {
        "epoch_loss": epoch_loss,
        "epoch_acc": epoch_acc,
        "train_step_losses": train_step_losses,
        "eval_step_losses": eval_step_losses,
        "best_f1": best_f1,
        "best_model_state": best_model_state,
        "best_report": best_report
    }

In [ ]:
EPOCHS = 10

And let's train the model!

In [ ]:
all_train_losses = []
all_eval_losses = []

best_f1_overall = 0.0
best_model_state_overall = None

for epoch in range(1, EPOCHS + 1):
    results = train_one_epoch(
        epoch,
        model,
        training_loader,
        validation_loader,
        optimizer,
        device,
        ids_to_labels,
        eval_every=10
    )

    # Extend training/evaluation losses
    all_train_losses.extend(results["train_step_losses"])
    all_eval_losses.extend(results["eval_step_losses"])

    # Display epoch summary
    print(f"\nEpoch {epoch} finished:")
    print(f"  Train loss: {results['epoch_loss']:.4f}")
    print(f"  Train acc:  {results['epoch_acc']:.4f}")
    print(f"  Best F1 in epoch: {results['best_f1']:.4f}")

    # Track overall best model
    if results["best_f1"] > best_f1_overall:
        best_f1_overall = results["best_f1"]
        best_model_state_overall = results["best_model_state"]
        print(f"  ✅ New overall best model (F1={best_f1_overall:.4f})")

# ✅ Load the best model after all epochs
if best_model_state_overall is not None:
    model.load_state_dict(best_model_state_overall)
    print(f"\n✅ Loaded best model (overall F1={best_f1_overall:.4f})")

In [ ]:
all_eval_losses

Save the model

In [ ]:
# ✅ Load the best model after all epochs
if best_model_state_overall is not None:
    model.load_state_dict(best_model_state_overall)
    print(f"\n✅ Loaded best model (overall F1={best_f1_overall:.4f})")

Let's see if there's any [underfitting/overfitting](https://www.geeksforgeeks.org/machine-learning/underfitting-and-overfitting-in-machine-learning/).

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("images", exist_ok=True)

plt.figure(figsize=(8,5))
plt.plot(all_train_losses, label='Training loss (per 10 steps)')
plt.plot(all_eval_losses, label='Eval loss (per 10 steps)', linestyle='--')
plt.xlabel("Steps (x10)")
plt.ylabel("Loss")
plt.title("Training vs Evaluation Loss over Steps")
plt.legend()

save_path = "images/training_train_eval_loss_run2.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()

print(f"Saved figure to: {save_path}")

#### **Evaluating the model**

Now that we've trained our model, we can evaluate its performance on the held-out test set (which is 20% of the data). Note that here, no gradient updates are performed, the model just outputs its logits.

In [ ]:
from seqeval.metrics import classification_report

labels, predictions = predict(model, testing_loader, device, ids_to_labels)

print(classification_report(labels, predictions, digits=4))



However, the accuracy metric is misleading, as a lot of labels are "outside" (O), even after omitting predictions on the [PAD] tokens. What is important is looking at the precision, recall and f1-score of the individual tags. For this, we use the seqeval Python library:

Performance already seems quite good, but note that we've only trained for 1 epoch. An optimal approach would be to perform evaluation on a validation set while training to improve generalization.

In [ ]:
valid_labels, valid_predictions = predict(model, validation_loader, device, ids_to_labels)
print(classification_report(valid_labels, valid_predictions, digits=4))

In [ ]:
import pandas as pd

def predictions_to_dataframe(test_data, predictions, entity_tags=("B-INSC", "I-INSC")):
    """
    Convert token-level predictions into a dataframe with extracted entity spans.

    Args:
        test_data (pd.DataFrame): DataFrame with a 'sentence' column (space-separated tokens).
        predictions (List[List[str]]): List of predicted tag sequences for each sentence.
        entity_tags (Tuple[str]): Tags to include when extracting entity tokens (default: B-INSC, I-INSC).

    Returns:
        pd.DataFrame: Columns ['sentence', 'predicted_inscriptions']
    """
    sentences, predicted_entities = [], []

    for idx, preds in enumerate(predictions):
        # Get sentence tokens
        tokens = test_data.sentence.iloc[idx].strip().split()
        min_len = min(len(tokens), len(preds))

        # Align to avoid mismatch
        tokens = tokens[:min_len]
        preds = preds[:min_len]

        # Extract tokens predicted as entity
        entity_tokens = [tok for tok, tag in zip(tokens, preds) if tag in entity_tags]

        sentences.append(" ".join(tokens))
        predicted_entities.append(" ".join(entity_tokens))

    df_pred = pd.DataFrame({
        "sentence": sentences,
        "predicted_inscriptions": predicted_entities
    })

    return df_pred

In [ ]:
df_predictions = predictions_to_dataframe(test_data, predictions)
df_predictions.head()

In [ ]:
# Extract all unique inscription terms from the training data
train_inscriptions = set()

for sentence, labels in zip(train_data.sentence, train_data.word_labels):
    tokens = sentence.strip().split()
    tags = labels.split(",")
    for token, tag in zip(tokens, tags):
        if tag in ("B-INSC", "I-INSC"):
            train_inscriptions.add(token)

print(f"{len(train_inscriptions)} unique inscription terms in training data")

In [ ]:
# Flatten all predicted inscriptions from your dataframe
predicted_inscriptions = set()

for insc_str in df_predictions["predicted_inscriptions"]:
    for token in insc_str.split():
        if token.strip():
            predicted_inscriptions.add(token)

print(f"{len(predicted_inscriptions)} unique inscription terms predicted")

In [ ]:
new_predicted_inscriptions = predicted_inscriptions - train_inscriptions

print(f"{len(new_predicted_inscriptions)} new inscription terms (not seen during training)")
print(new_predicted_inscriptions) # false positives

```
'donor', 'to do'
```

## Augmented Dataset

We need to *re-load* the model:

In [ ]:
# model = BertForTokenClassification.from_pretrained(BASE_MODEL, num_labels=len(labels_to_ids))
# model.to(device)

In [ ]:
# train_data_aug = pd.read_csv('dataset_formation/data/train_ner_aug_doctored.csv')
# # eval_data_aug= pd.read_csv('dataset_formation/data/eval_ner_aug_doctored.csv')

# eval_data_aug = pd.read_csv('dataset_formation/data/eval_ner_doctored.csv') # eval simulates a real test
# test_data_aug = pd.read_csv('dataset_formation/data/test_ner_doctored.csv')


In [ ]:

# training_set_aug = dataset(train_data_aug, tokenizer, MAX_LEN)
# validation_set_aug = dataset(eval_data_aug, tokenizer, MAX_LEN)

In [ ]:
# training_loader_aug = DataLoader(training_set_aug, **train_params)
# # validation_loader_aug = DataLoader(validation_set_aug, **test_params)

In [ ]:
# for epoch in range(EPOCHS):
#     print(f"Training epoch: {epoch + 1}")
#     train(epoch,training_loader=training_loader_aug)

In [ ]:
# all_train_losses = []
# all_eval_losses = []

# for epoch in range(1, EPOCHS + 1):
#     train_loss, train_acc, step_train_losses, step_eval_losses = train_one_epoch(
#         epoch, model, training_loader_aug, validation_loader, optimizer, device, ids_to_labels, eval_every=100
#     )
#     all_train_losses.extend(step_train_losses)
#     all_eval_losses.extend(step_eval_losses)

Save the model..

In [ ]:
# code for saving the model

In [ ]:
# plt.figure(figsize=(8,5))
# plt.plot(all_train_losses, label='Training loss (per 100 steps)')
# plt.plot(all_eval_losses, label='Eval loss (per 100 steps)', linestyle='--')
# plt.xlabel("Steps (x100)")
# plt.ylabel("Loss")
# plt.title("Training vs Evaluation Loss over Steps")
# plt.legend()
# plt.show()

In [ ]:
# test_aug_labels, test_aug_predictions = predict(model, testing_loader, device, ids_to_labels)
# print(classification_report(test_aug_labels, test_aug_predictions, digits=4))

In [ ]:
# test_aug_predictions[:5]

In [ ]:
# test_aug_labels[:5]

In [ ]:
# valid_labels, valid_predictions = predict(model, validation_loader, device, ids_to_labels)
# print(classification_report(valid_labels, valid_predictions, digits=4))

In [ ]:
# df_aug_predictions = predictions_to_dataframe(test_data, test_aug_predictions)
# df_aug_predictions.head()

In [ ]:
# df_predictions.predicted_inscriptions.head()


In [ ]:
# labels, predictions = valid_rough(model, testing_loader,device,ids_to_labels)
# print(classification_report(labels, predictions,digits=4))